In [ ]:
import os
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as op
import optuna
import pickle
# custom
from h5dataset import h5set
from automobili import EricGio
from funkytrain import traingio, evalgio

data_train = torch.load("/mnt/eph/data/train_test_val/train.pt", weights_only=False)
training = DataLoader(data_train, batch_size=1, num_workers=os.cpu_count())

data_val = torch.load("/mnt/eph/data/train_test_val/val.pt", weights_only=False)
validation = DataLoader(data_val, batch_size=1, num_workers=os.cpu_count())


artifact_store = optuna.artifacts.FileSystemArtifactStore(base_path='/mnt/eph/artifacts')

def objective(trial:optuna.Trial):
    num_epochs = 50
    loss_func = nn.MSELoss()
    train_loader = training
    test_loader = validation
    
    # Trial choices
    # activation = trial.suggest_categorical("activation", ['ReLU', 'LeakyReLU'])
    optim = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'AdamW', 'RMSprop', 'Adagrad', 'NAdam'])
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    
    # Training phase
    # activation = getattr(nn, activation)
    model = EricGio().to(device)
    if optim in ('SGD', "RMSprop"):
        momentum = trial.suggest_float("momentum", 0.3, 0.9, step=0.1)
        optim = getattr(op, optim)(model.parameters(), lr=lr, momentum=momentum)
    else: optim = getattr(op, optim)(model.parameters(), lr=lr)
    
    losses = {'train':[], 'test':[]}
    for epoch in range(1, num_epochs + 1):
        train_loss = traingio(model, device, train_loader, loss_func, optim)

        val_loss = evalgio(model=model, device=device, dataloader=test_loader, loss_fn=loss_func)
        print(f'TRAIN - EPOCH {epoch}/{num_epochs} - loss: {train_loss} - test loss: {val_loss}')
        trial.report(val_loss, epoch)


        if trial.should_prune(): raise optuna.TrialPruned()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optim_state_dict': optimizer.state_dict()
    }, 'model.pt')

    torch.save(model.loss_dict, 'model.pyd')

    art_id = optuna.artifacts.upload_artifact(artifact_store=artifact_store, file_path='model.pt', study_or_trial=trial.study)
    dict_id = optuna.artifacts.upload_artifact(artifact_store=artifact_store, file_path='model.pyd', study_or_trial=trial.study)

    trial.set_user_attr('model_optim_id', art_id)
    trial.set_user_attr('dict_loss_id', dict_id)


    
    return val_loss


optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
# to change model, just change the study name, don't touch storage
study = optuna.create_study(direction='minimize', study_name='Eric', storage=f"sqlite:///CanOTuna.db", load_if_exists=True)
study.optimize(objective, n_trials=30)